In [1]:
import requests
import numpy as np
import pandas as pd
from datetime import date
import os
import time
import random
import duckdb
DB_PATH = r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\duckdb\nhl_data.duckdb"
con = duckdb.connect(DB_PATH) 
print("Connected:", DB_PATH)
pd.options.mode.chained_assignment = None

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

Connected: C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\duckdb\nhl_data.duckdb


In [3]:
con.close()
print("DuckDB connection closed.")

DuckDB connection closed.


In [ ]:
# con.execute("""

# CREATE TABLE IF NOT EXISTS team_daily_analytics AS
# SELECT * FROM teams

# """)

In [8]:


con.execute("SHOW TABLES").fetchdf()

,name
0,all_clinch_history
1,all_final_standings
2,all_game_summary
3,all_game_summary_backup
4,all_goalie_games
5,all_pk
6,all_play_by_play
7,all_play_by_play_1
8,all_pp
9,all_prediction_data_agg


In [2]:


con.execute("""
select *
from all_game_summary
where season_id = 20252026
order by game_id
""").fetch_df().tail()

,game_id,season_id,away_team,away_tri_code,home_team,home_tri_code,awayTeam.score,homeTeam.score,gameState,link,game_date,api_snapshot_date
1307,2025021308,20252026,28,SJS,52,WPG,<NA>,<NA>,FUT,https://api-web.nhle.com/v1/gamecenter/2025021308/play-by-play,2026-04-16,2025-12-09
1308,2025021309,20252026,19,STL,68,UTA,<NA>,<NA>,FUT,https://api-web.nhle.com/v1/gamecenter/2025021309/play-by-play,2026-04-16,2025-12-09
1309,2025021310,20252026,26,LAK,20,CGY,<NA>,<NA>,FUT,https://api-web.nhle.com/v1/gamecenter/2025021310/play-by-play,2026-04-16,2025-12-09
1310,2025021311,20252026,23,VAN,22,EDM,<NA>,<NA>,FUT,https://api-web.nhle.com/v1/gamecenter/2025021311/play-by-play,2026-04-16,2025-12-09
1311,2025021312,20252026,55,SEA,21,COL,<NA>,<NA>,FUT,https://api-web.nhle.com/v1/gamecenter/2025021312/play-by-play,2026-04-16,2025-12-09


In [ ]:
# con.execute("""

# CREATE TABLE IF NOT EXISTS team_daily_stats_backup AS
# SELECT * FROM team_daily_stats

# """)

# con.execute("ALTER TABLE team_daily_stats_backup RENAME TO team_daily_stats;")
# con.execute("DROP TABLE IF EXISTS playoff_daily_predictions")

In [3]:
daily_predictions = con.execute("""
select *
from playoff_daily_predictions
order by game_date
""").fetch_df()

In [ ]:

LEAK_COLS = [
    "game_id", "game_date", "date",
    "clinch_type", "made_playoffs",
    "team_name", "opponent",
    "season_id"
]

ID_COLS = ["team_id", "opponent_id", "tri_code"]

X_shap = (
    daily_predictions
    .drop(columns=LEAK_COLS + ID_COLS, errors="ignore")
    .select_dtypes(include="number")
    .fillna(0)
    .reindex(columns=FEATURE_COLS, fill_value=0)
)

# =====================================================
# 3. APPLY SCALER (what the model actually sees)
# =====================================================

X_shap_scaled = scaler.transform(X_shap)

X_shap_scaled = pd.DataFrame(
    X_shap_scaled,
    columns=X_shap.columns,
    index=X_shap.index
)

In [13]:
con.execute("""
select *
from team_daily_stats
where season_id = 20252026 and team_id = 6
order by game_id
""").fetch_df().tail(10)

,season_id,game_id,game_date,team_id,tri_code,team_name,game_win,wins_before,games_before,win_pct_before,sos_past,sos_future,pp_opp,pp_opp_pg,pp_toi,pp_gf,pp_gpg,pp_net_g,pp_net_gpg,pp_pct,pp_net_pct,sh_ga,sh_ga_pg,times_sh,times_sh_pg,pk_net_pct,pk_pct,pk_net_g,pk_net_gpg,pk_toi,pp_ga,pp_ga_pg,sh_gf,sh_gf_pg,c_pp_opps,c_pp_net_g,ppng_rank,c_pp_ga,c_times_sh,pk_rank,game_savePct,game_GAA,rush_shots_for,rush_shots_against,rush_shot_differential,c_rush_shots_for,c_rush_shots_against,c_rush_diff,rush_for_per_game,rush_against_per_game,rush_diff_per_game,shot_attempts_for,shot_attempts_against,shot_attempts_differential,corsi_for,corsi_against,corsi_diff,c_shots_diff,games_played,shot_attempts_for_per_game,shot_attempts_against_per_game,shots_diff_per_game,SCF,SCSF,SCGF,SCA,HDCF,MDCF,LDCF,HDCA,MDCA,LDCA,HDCF_pct,HDCA_pct,MDCF_pct,MDCA_pct,LDCF_pct,LDCA_pct,c_SCF,c_SCSF,c_SCGF,c_SCA,SCF_pg,SCA_pg,c_HDCF,c_HDCA,c_MDCF,c_MDCA,c_LDCF,c_LDCA,shots_for,hd_sog,md_sog,ld_sog,hd_goals,md_goals,ld_goals,shot_pct,shots_against,hd_against,md_against,ld_against,sv_pct,pdo,c_goals_for,c_hd_sog,c_md_sog,c_ld_sog,c_hd_goals,c_md_goals,c_ld_goals,c_goals_against,c_hd_against,c_md_against,c_ld_against,c_shot_pct,c_sv_pct,c_pdo,c_shots_for,c_shots_against,c_corsi_for,c_corsi_against,c_corsi_diff,corsi_diff_per_game,games_remaining,wins,wins_pctg,losses,ot_losses,rotw,rotw_pctg,rw,so_losses,so_wins,points,points_pct,projected_points,in_playoff_spot,gf_pctg,goal_diff,goal_diff_pctg,streak,streak_count,last_10_gp,last_10_wins,last_10_losses,last_10_otl,last_10_points,last_10_gf,last_10_ga,last_10_goal_diff,last_10_rotw,last_10_rw,home_gp,home_wins,home_losses,home_otlosses,home_rotw,home_rw,home_points,home_gf,home_ga,home_goal_diff,road_gp,road_wins,road_losses,road_ot_losses,road_rotw,road_rw,road_points,road_gf,road_ga,road_goal_diff,division,division_rank,division_home_rank,division_road_rank,division_last_10_rank,conference,conference_rank,conference_home_rank,conference_road_rank,conference_last_10_rank,league_rank,league_home_rank,league_road_rank,league_last_10_rank,wildcard_rank,made_playoffs,goals_for,goals_against,ot_wins,overtimes,ot_games_pct,pt_pct_x_sos,gaa_x_sv,gd_x_rushdiff,pk_load_x_pk_pct
47,20252026,2025020739,2026-01-15,6,BOS,Boston Bruins,1,26,47,0.553191,0.474283,0.496987,2,2.958333,169.0,1,0.020833,1,0.625000,50.0,21.126761,0,0.000000,4,3.729167,21.2,78.8,-1,-0.020833,359.0,2,0.791667,1,0.020833,142.0,30.0,6,38.0,179.0,11,0.928570,2.000000,4,4,0,353,339,14,7.354167,7.062500,0.291667,53,57,-4,33,41,-8,-68,48,60.000000,61.416667,-1.416667,23,24,4,37,13,11,30,23,14,20,0.361111,0.638889,0.440000,0.560000,0.600000,0.400000,920,1326,159,1957,31.312500,32.645833,419,419,544,544,1134,1134,24,4,5,15,2,1,1,16.666667,28,12,4,12,92.857143,109.523810,159,274,358,718,60,62,43,148,379,382,715,0.056584,0.948107,100.469022,2880,2948,1994,2133,-139,-2.895833,34,27,0.562500,19,2,25,0.520833,20,1,2,56,0.583333,96,0,3.312500,11,0.229167,W,5.0,10,7,2,1,15,37,22,15,7,6,26,17,8,1,16,13,35,90,72,18,22,10,11,1,9,7,21,69,76,-7,Atlantic,5,2,6,3,Eastern,7,2,14,3,11,3,24,3,2,<NA>,4,2,5,7,0.15,0.289909,1.857140,0,293.858333
48,20252026,2025020763,2026-01-17,6,BOS,Boston Bruins,1,27,48,0.562500,0.472824,0.500356,1,2.918367,86.0,1,0.040816,1,0.632653,100.0,21.678322,0,0.000000,1,3.673469,21.1,78.9,0,-0.020408,120.0,0,0.775510,0,0.020408,143.0,31.0,5,38.0,180.0,12,0.916660,2.000000,8,5,3,361,344,17,7.367347,7.020408,0.346939,51,54,-3,36,43,-7,-71,49,59.816327,61.265306,-1.448980,28,23,5,26,16,16,23,13,15,28,0.551724,0.448276,0.516129,0.483871,0.450980,0.549020,937,1349,164,1974,31.244898,32.510204,430,430,553,553,1154,1154,23,8,8,10,3,2,2,21.739130,24,5,6,14,91.666667,113.405797,164,282,366,728,63,64,45,150,384,388,729,0.057889,0.947844,100.573339,2931,3002,2030,2176,-146,-2.979592,33,28,0.571429,19,2,26,0.530612,21,1,2,58,0.591837,97,0,3.346939,14,0.285714,W,6.0,10,8,1,1,17,41,20,21,8,7,26,17,8,1,16,13,35,90,72,18,23,11,11,1,10,8,23,74,78,-4,Atlantic,4,3,5,2,Ea

In [6]:
con.execute("""
select *
from team_daily_stats
""").fetch_df().head()

,season_id,game_id,game_date,team_id,tri_code,team_name,game_win,wins_before,games_before,win_pct_before,sos_past,sos_future,pp_opp,pp_opp_pg,pp_toi,pp_gf,pp_gpg,pp_net_g,pp_net_gpg,pp_pct,pp_net_pct,sh_ga,sh_ga_pg,times_sh,times_sh_pg,pk_net_pct,pk_pct,pk_net_g,pk_net_gpg,pk_toi,pp_ga,pp_ga_pg,sh_gf,sh_gf_pg,c_pp_opps,c_pp_net_g,ppng_rank,c_pp_ga,c_times_sh,pk_rank,game_savePct,game_GAA,rush_shots_for,rush_shots_against,rush_shot_differential,c_rush_shots_for,c_rush_shots_against,c_rush_diff,rush_for_per_game,rush_against_per_game,rush_diff_per_game,shot_attempts_for,shot_attempts_against,shot_attempts_differential,corsi_for,corsi_against,corsi_diff,c_shots_diff,games_played,shot_attempts_for_per_game,shot_attempts_against_per_game,shots_diff_per_game,SCF,SCSF,SCGF,SCA,HDCF,MDCF,LDCF,HDCA,MDCA,LDCA,HDCF_pct,HDCA_pct,MDCF_pct,MDCA_pct,LDCF_pct,LDCA_pct,c_SCF,c_SCSF,c_SCGF,c_SCA,SCF_pg,SCA_pg,c_HDCF,c_HDCA,c_MDCF,c_MDCA,c_LDCF,c_LDCA,shots_for,hd_sog,md_sog,ld_sog,hd_goals,md_goals,ld_goals,shot_pct,shots_against,hd_against,md_against,ld_against,sv_pct,pdo,c_goals_for,c_hd_sog,c_md_sog,c_ld_sog,c_hd_goals,c_md_goals,c_ld_goals,c_goals_against,c_hd_against,c_md_against,c_ld_against,c_shot_pct,c_sv_pct,c_pdo,c_shots_for,c_shots_against,c_corsi_for,c_corsi_against,c_corsi_diff,corsi_diff_per_game,games_remaining,wins,wins_pctg,losses,ot_losses,rotw,rotw_pctg,rw,so_losses,so_wins,points,points_pct,projected_points,in_playoff_spot,gf_pctg,goal_diff,goal_diff_pctg,streak,streak_count,last_10_gp,last_10_wins,last_10_losses,last_10_otl,last_10_points,last_10_gf,last_10_ga,last_10_goal_diff,last_10_rotw,last_10_rw,home_gp,home_wins,home_losses,home_otlosses,home_rotw,home_rw,home_points,home_gf,home_ga,home_goal_diff,road_gp,road_wins,road_losses,road_ot_losses,road_rotw,road_rw,road_points,road_gf,road_ga,road_goal_diff,division,division_rank,division_home_rank,division_road_rank,division_last_10_rank,conference,conference_rank,conference_home_rank,conference_road_rank,conference_last_10_rank,league_rank,league_home_rank,league_road_rank,league_last_10_rank,wildcard_rank,made_playoffs,goals_for,goals_against,ot_wins,overtimes,ot_games_pct,pt_pct_x_sos,gaa_x_sv,gd_x_rushdiff,pk_load_x_pk_pct,pp_opp_pg_1,pp_gpg_1,pp_net_gpg_1,sh_ga_pg_1,times_sh_pg_1,pk_net_gpg_1,pp_ga_pg_1,sh_gf_pg_1
0,20132014,2013020008,2013-10-03,1,NJD,New Jersey Devils,0,<NA>,0,NaN,NaN,0.491552,2,2.0,155.0,0,0.0,0,0.0,0.00000,0.000000,0,0.0,1,1.0,1.0,1.00,0,0.0,120.0,0,0.0,0,0.0,2.0,0.0,10,0.0,1.0,1,0.85714,3.00000,6,4,2,6,4,2,6.000000,4.000000,2.000000,59,58,1,35,37,-2,1,1,59.000000,58.000000,1.000000,29,27,0,58,15,26,21,13,19,17,0.535714,0.464286,0.577778,0.422222,0.552632,0.447368,29,27,0,58,29.000000,58.000000,15,13,26,19,21,17,27,7,9,13,0,0,0,0.000000,21,2,9,10,85.714286,85.714286,0,7,9,13,0,0,0,3,2,9,10,0.000000,0.857143,85.714286,59,58,35,37,-2,-2.0,81,0,0.0,1,0,0,0.0,0,0,0,0,0.000000,0,0,0.000000,-3,-3.000000,L,1.0,1,0,1,0,0,0,3,-3,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,3,-3,Metropolitan,7,5,7,7,Eastern,15,9,15,15,27,19,27,27,9,0,0,3,0,0,0.00,0.000000,2.571420,-6,1.0,2.000000,0.000000,0.000000,0.0,1.000000,0.0,0.000000,0.000000
1,20132014,2013020016,2013-10-04,1,NJD,New Jersey Devils,0,0,1,0.0,NaN,0.491552,2,2.0,240.0,0,0.0,0,0.0,0.00000,0.000000,0,0.0,2,2.0,1.0,1.00,0,0.0,240.0,0,0.0,0,0.0,4.0,0.0,12,0.0,3.0,1,0.88461,2.76923,7,9,-2,13,13,0,6.500000,6.500000,0.000000,61,61,0,47,50,-3,1,2,60.000000,59.500000,0.500000,33,34,3,63,22,17,24,20,15,23,0.523810,0.476190,0.531250,0.468750,0.510638,0.489362,62,61,3,121,31.000000,60.500000,37,33,43,34,45,40,34,16,8,11,2,1,0,8.823529,30,11,9,11,86.666667,95.490196,3,23,17,24,2,1,0,7,13,18,21,0.049180,0.862745,91.192543,120,119,82,87,-5,-2.5,80,0,0.0,1,1,0,0.0,0,1,0,1,0.250000,41,0,1.500000,-4,-2.000000,OT,1.0,2,0,1,1,1,3,7,-4,0,0,1,0,0,1,0,0,1,3,4,-1,1,0,1,0,0,0,0,0,3,-3,Metropolitan,5,4,7,5,Eastern,10,6,15,10,19,13,26,19,4,0,3,4,0,1,0.50,0.122888,2.449689,8,2.0,1.000000,0.000000,0.000000,0.0,1.000000,0.0,0.000000,0.0

In [ ]:

# con.execute("""SELECT
#     game_id,
#     COUNT(*) AS cnt
# FROM all_game_summary
# GROUP BY game_id
# HAVING COUNT(*) > 1
# ORDER BY cnt DESC;
# """).fetchdf()


,game_id,cnt


In [ ]:

con.register("teams_updates", teams)


In [ ]:

# con.execute("""
#     INSERT INTO all_teams (team_id, tri_code)
#     SELECT
#         u.team_id,
#         u.tri_code
#     FROM teams_updates u
#     LEFT JOIN all_teams t
#       ON t.team_id = u.team_id
#     WHERE t.team_id IS NULL;
# """)


In [2]:
# con.execute("ALTER TABLE prediction_data_backup DROP COLUMN net_pp_pct").fetchdf()
# # con.execute("""ALTER TABLE prediction_data_backup 
# # RENAME COLUMN pk_toi_pg TO pk_toi;""")
# con.execute("DESCRIBE prediction_data_backup").fetchdf()

In [ ]:
# def build_clinch_history_archive():

  
  

# base_url = "https://api-web.nhle.com/v1/standings/"

# all_clinch_rows = []

  
#     for season, end_date_str in season_end_dates.items():

# print(f"Processing season {season}")

#         end_date = pd.to_datetime(end_date_str)

#         start_date = pd.to_datetime(f"{end_date.year}-03-01")

#         today = end_date

#         last_ran_date = start_date

  

#         team_status = {}

  

#         for check_date in pd.date_range(start=start_date, end=end_date):

#             url = f"{base_url}{check_date.strftime('%Y-%m-%d')}"

#             try:

#                 response = requests.get(url)

#                 if response.status_code != 200:

#                     continue

  

#                 standings_data = response.json()["standings"]

  

#                 # Group by conference

#                 east_teams = [t for t in standings_data if t["conferenceName"] == "Eastern"]

#                 west_teams = [t for t in standings_data if t["conferenceName"] == "Western"]

  

#                 for conference_teams in [east_teams, west_teams]:

#                     # Grab the team in 8th spot using "conferenceSequence"

#                     eighth_place_team = next((t for t in conference_teams if t["conferenceSequence"] == 8), None)

#                     cutoff_points = eighth_place_team["points"] if eighth_place_team else 0

  
  

#                     for team in conference_teams:

#                         tri_code = team["teamAbbrev"]["default"]

#                         games_played = team["gamesPlayed"]

#                         points = team["points"]

#                         clinch = team.get("clinchIndicator", None)

  

#                         if tri_code not in team_status:

#                             team_status[tri_code] = {

#                                 "last_ran_date": last_ran_date.date(),

#                                 "clinch_indicator": False,

#                                 "tri_code": tri_code,

#                                 "clinch_type": None,

#                                 "clinch_date": None,

#                                 "season_end": today.date(),

#                                 "season": season

#                             }

  

#                         if clinch and team_status[tri_code]["clinch_type"] is None:

#                             team_status[tri_code]["clinch_type"] = clinch

#                             team_status[tri_code]["clinch_indicator"] = True

#                             team_status[tri_code]["clinch_date"] = check_date.date()

#                             continue  # already clinched

  

#                         # Check for elimination

#                         total_games = 82

#                         remaining_games = total_games - games_played

#                         max_possible_points = points + (remaining_games * 2)

  

#                         if team_status[tri_code]["clinch_type"] is None and max_possible_points < cutoff_points:

#                             team_status[tri_code]["clinch_type"] = "e"

#                             team_status[tri_code]["clinch_indicator"] = True

#                             team_status[tri_code]["clinch_date"] = check_date.date()

  

#             except Exception as e:

#                 print(f"Error on {check_date.date()} for {season}: {e}")

  

#         # Backfill unmarked teams (just in case)

#         for tri_code, data in team_status.items():

#             if data["clinch_type"] is None:

#                 data["clinch_type"] = 'e'

#                 data["clinch_indicator"] = True

#                 data["clinch_date"] = end_date.date()

#             all_clinch_rows.append(data)

  

#     clinch_history_df = pd.DataFrame(all_clinch_rows)

#     clinch_history_df = clinch_history_df[

#         ['last_ran_date', 'clinch_indicator', 'tri_code', 'clinch_type', 'clinch_date', 'season_end', 'season']

#     ].sort_values(by=["season", "clinch_date", "tri_code"]).reset_index(drop=True)

  

#     return clinch_history_df